# GGA Network Training Example

This notebook demonstrates how to train a GGA-level exchange-correlation functional using xcquinox.

## Outline

1. **Setup**: Import libraries and define helper functions
2. **Pre-training**: Train exchange (X) and correlation (C) networks on reference functional data (PBE)
3. **Training Part I**: Train on total energies of atoms + atomization energies of small molecules
4. **Training Part II**: Include density error in the loss function

All calculations use PySCF-AD (autodifferentiable PySCF) as the calculation driver.

---
## 1. Setup

In [ ]:
# Core imports
import numpy as np
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import matplotlib.pyplot as plt

# PySCF imports
import pyscf
from pyscf import gto, dft, scf

# PySCF-AD imports (autodifferentiable)
from pyscfad import gto as gtoad
from pyscfad import dft as dftad

# xcquinox imports
import xcquinox as xce
from xcquinox.pyscf import eval_xc_gga_pol
from functools import partial

# ASE for molecular structures
from ase import Atoms

print(f"JAX version: {jax.__version__}")
print(f"PySCF version: {pyscf.__version__}")
print(f"Using device: {jax.devices()[0]}")

In [ ]:
# Configuration - keep computations affordable
BASIS = 'sto-3g'           # Small basis for speed
GRID_LEVEL = 1             # Coarse grid for speed
REFERENCE_XC = 'PBE'       # Reference functional for pre-training

# Use CPU to avoid GPU memory issues
CPU_DEVICE = jax.devices('cpu')[0]

# Spin configurations for isolated atoms
SPINS_DICT = {
    'H': 1, 'He': 0, 'Li': 1, 'Be': 0, 'B': 1, 'C': 2, 'N': 3, 'O': 2, 'F': 1, 'Ne': 0,
    'Na': 1, 'Mg': 0, 'Al': 1, 'Si': 2, 'P': 3, 'S': 2, 'Cl': 1, 'Ar': 0
}

In [ ]:
def create_mol(atoms_str, basis=BASIS, charge=0, spin=None):
    '''
    Create a PySCF Mole object from atom string.
    
    :param atoms_str: Atom specification (e.g., 'H 0 0 0; H 0 0 0.74')
    :param basis: Basis set name
    :param charge: Molecular charge
    :param spin: 2S (number of unpaired electrons), auto-detected for single atoms
    :return: Built PySCF Mole object
    '''
    mol = gto.Mole()
    mol.atom = atoms_str
    mol.basis = basis
    mol.charge = charge
    if spin is not None:
        mol.spin = spin
    mol.build()
    return mol


def create_mol_ad(atoms_str, basis=BASIS, charge=0, spin=None):
    '''
    Create a PySCF-AD Mole object (autodifferentiable).
    '''
    mol = gtoad.Mole()
    mol.atom = atoms_str
    mol.basis = basis
    mol.charge = charge
    if spin is not None:
        mol.spin = spin
    mol.build()
    return mol


def run_pbe_calculation(mol):
    '''
    Run a standard PBE DFT calculation.
    
    :param mol: PySCF Mole object
    :return: Converged mf object
    '''
    if mol.spin == 0:
        mf = dft.RKS(mol)
    else:
        mf = dft.UKS(mol)
    mf.xc = REFERENCE_XC
    mf.grids.level = GRID_LEVEL
    mf.kernel()
    return mf


def get_enhancement_factor_data(mol, mf, xorc='x'):
    '''
    Extract enhancement factor data from a converged calculation for pre-training.
    
    :param mol: PySCF Mole object
    :param mf: Converged mf object
    :param xorc: 'x' for exchange or 'c' for correlation
    :return: (descriptors, reference_enhancement_factors)
    '''
    ao = mf._numint.eval_ao(mol, mf.grids.coords, deriv=1)
    dm = mf.make_rdm1()
    
    # Handle spin
    if len(dm.shape) == 2:
        dm = np.array([0.5*dm, 0.5*dm])
    
    # Evaluate density on grid
    rho_alpha = mf._numint.eval_rho(mol, ao, dm[0], xctype='GGA', hermi=True)
    rho_beta = mf._numint.eval_rho(mol, ao, dm[1], xctype='GGA', hermi=True)
    
    if xorc == 'x':
        # Exchange: Fx = exc / lda_x
        xc_func = f'{REFERENCE_XC},'
        exc = mf._numint.eval_xc(xc_func, (rho_alpha, rho_alpha*0), spin=1)[0]
        lda_exc = mf._numint.eval_xc('LDA_X,', (rho_alpha, rho_alpha*0), spin=1)[0]
        Fxc = exc / (lda_exc + 1e-10) - 1  # Enhancement factor minus 1
    else:
        # Correlation: Fc = ec / lda_c
        xc_func = f',{REFERENCE_XC}'
        exc = mf._numint.eval_xc(xc_func, (rho_alpha, rho_beta), spin=1)[0]
        lda_exc = mf._numint.eval_xc(',LDA_C_PW', (rho_alpha, rho_beta), spin=1)[0]
        Fxc = exc / (lda_exc + 1e-10) - 1
    
    # Build descriptors: rho, sigma -> s (reduced density gradient)
    rho0 = rho_alpha[0]
    drho = rho_alpha[1:4]
    sigma = np.sum(drho**2, axis=0)
    
    # Filter out low-density regions
    valid = rho0 > 1e-6
    rho0 = rho0[valid]
    sigma = sigma[valid]
    Fxc = Fxc[valid]
    
    # Compute reduced density gradient s
    k_F = (3 * np.pi**2 * rho0)**(1/3)
    s = np.sqrt(sigma) / (2 * k_F * rho0 + 1e-10)
    
    # Stack as input features [rho, sigma] or just [s] depending on network
    descriptors = np.stack([rho0, sigma], axis=1)
    
    return jnp.array(descriptors), jnp.array(Fxc)

---
## 2. Pre-training

Pre-training fits the neural network enhancement factors (Fx and Fc) to reproduce a reference functional (PBE) on a set of atoms and small molecules.

We'll explore two different network architectures:
- **Architecture A**: Shallow network (depth=2, nodes=8)
- **Architecture B**: Deeper network (depth=3, nodes=16)

In [ ]:
# Define pre-training molecules (small atoms and diatomics)
pretrain_systems = [
    ('H', 1),      # Hydrogen atom
    ('C', 2),      # Carbon atom
    ('N', 3),      # Nitrogen atom
    ('O', 2),      # Oxygen atom
    ('H 0 0 0; H 0 0 0.74', 0),   # H2
    ('N 0 0 0; N 0 0 1.1', 0),     # N2
    ('O 0 0 0; O 0 0 1.21', 2),    # O2 (triplet)
]

print("Pre-training molecules:")
for atoms, spin in pretrain_systems:
    print(f"  {atoms} (spin={spin})")

In [ ]:
# Collect enhancement factor data from reference calculations
print("Running reference calculations and collecting enhancement factor data...")

all_descriptors_x = []
all_Fx = []
all_descriptors_c = []
all_Fc = []

for atoms_str, spin in pretrain_systems:
    print(f"  Processing: {atoms_str}")
    mol = create_mol(atoms_str, spin=spin)
    mf = run_pbe_calculation(mol)
    
    # Exchange data
    desc_x, Fx = get_enhancement_factor_data(mol, mf, xorc='x')
    all_descriptors_x.append(desc_x)
    all_Fx.append(Fx)
    
    # Correlation data
    desc_c, Fc = get_enhancement_factor_data(mol, mf, xorc='c')
    all_descriptors_c.append(desc_c)
    all_Fc.append(Fc)

# Concatenate all data
train_desc_x = jnp.concatenate(all_descriptors_x, axis=0)
train_Fx = jnp.concatenate(all_Fx, axis=0)
train_desc_c = jnp.concatenate(all_descriptors_c, axis=0)
train_Fc = jnp.concatenate(all_Fc, axis=0)

print(f"\nCollected data shapes:")
print(f"  Exchange descriptors: {train_desc_x.shape}")
print(f"  Exchange Fx targets: {train_Fx.shape}")
print(f"  Correlation descriptors: {train_desc_c.shape}")
print(f"  Correlation Fc targets: {train_Fc.shape}")

In [ ]:
class PretrainLoss(eqx.Module):
    '''
    MSE loss for pre-training enhancement factor networks.
    '''
    def __call__(self, model, descriptors, ref_F):
        '''
        Compute MSE loss between predicted and reference enhancement factors.
        
        :param model: GGA network (GGA_FxNet_sigma or GGA_FcNet_sigma)
        :param descriptors: Input descriptors [rho, sigma]
        :param ref_F: Reference enhancement factors
        :return: MSE loss
        '''
        # Network expects [rho, sigma] and outputs Fx or Fc
        pred = jax.vmap(model)(descriptors)
        # Subtract 1 because networks output 1 + enhancement
        pred = pred - 1.0
        return jnp.mean((pred - ref_F)**2)

### Architecture A: Shallow Network (depth=2, nodes=8)

In [ ]:
# Create shallow networks
DEPTH_A = 2
NODES_A = 8
SEED = 42

# Exchange network
xnet_A = xce.net.GGA_FxNet_sigma(depth=DEPTH_A, nodes=NODES_A, seed=SEED)
print(f"Exchange Network A: depth={DEPTH_A}, nodes={NODES_A}")

# Correlation network
cnet_A = xce.net.GGA_FcNet_sigma(depth=DEPTH_A, nodes=NODES_A, seed=SEED)
print(f"Correlation Network A: depth={DEPTH_A}, nodes={NODES_A}")

In [ ]:
# Pre-train Architecture A
print("Pre-training Architecture A...")

PRETRAIN_STEPS = 500
PRETRAIN_LR = 1e-2

# Learning rate schedule with decay
scheduler = optax.exponential_decay(
    init_value=PRETRAIN_LR,
    transition_begin=50,
    transition_steps=200,
    decay_rate=0.9
)
optimizer = optax.adam(learning_rate=scheduler)
loss_fn = PretrainLoss()

# Train exchange network
print("\n--- Training Exchange Network A ---")
trainer_x = xce.train.xcTrainer(
    model=xnet_A,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    xnet_A_trained = trainer_x(1, xnet_A, [train_desc_x], [train_Fx])

# Train correlation network
print("\n--- Training Correlation Network A ---")
trainer_c = xce.train.xcTrainer(
    model=cnet_A,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    cnet_A_trained = trainer_c(1, cnet_A, [train_desc_c], [train_Fc])

### Architecture B: Deeper Network (depth=3, nodes=16)

In [ ]:
# Create deeper networks
DEPTH_B = 3
NODES_B = 16

# Exchange network
xnet_B = xce.net.GGA_FxNet_sigma(depth=DEPTH_B, nodes=NODES_B, seed=SEED)
print(f"Exchange Network B: depth={DEPTH_B}, nodes={NODES_B}")

# Correlation network  
cnet_B = xce.net.GGA_FcNet_sigma(depth=DEPTH_B, nodes=NODES_B, seed=SEED)
print(f"Correlation Network B: depth={DEPTH_B}, nodes={NODES_B}")

In [ ]:
# Pre-train Architecture B
print("Pre-training Architecture B...")

# Train exchange network
print("\n--- Training Exchange Network B ---")
trainer_x_B = xce.train.xcTrainer(
    model=xnet_B,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    xnet_B_trained = trainer_x_B(1, xnet_B, [train_desc_x], [train_Fx])

# Train correlation network
print("\n--- Training Correlation Network B ---")
trainer_c_B = xce.train.xcTrainer(
    model=cnet_B,
    optim=optimizer,
    steps=PRETRAIN_STEPS,
    loss=loss_fn,
    do_jit=True
)

with jax.default_device(CPU_DEVICE):
    cnet_B_trained = trainer_c_B(1, cnet_B, [train_desc_c], [train_Fc])

In [ ]:
# Compare pre-training results
print("\n" + "="*60)
print("Pre-training Results Comparison")
print("="*60)

# Evaluate on training data
pred_Fx_A = jax.vmap(xnet_A_trained)(train_desc_x) - 1.0
pred_Fx_B = jax.vmap(xnet_B_trained)(train_desc_x) - 1.0
pred_Fc_A = jax.vmap(cnet_A_trained)(train_desc_c) - 1.0
pred_Fc_B = jax.vmap(cnet_B_trained)(train_desc_c) - 1.0

rmse_Fx_A = jnp.sqrt(jnp.mean((pred_Fx_A - train_Fx)**2))
rmse_Fx_B = jnp.sqrt(jnp.mean((pred_Fx_B - train_Fx)**2))
rmse_Fc_A = jnp.sqrt(jnp.mean((pred_Fc_A - train_Fc)**2))
rmse_Fc_B = jnp.sqrt(jnp.mean((pred_Fc_B - train_Fc)**2))

print(f"\nExchange Enhancement Factor RMSE:")
print(f"  Architecture A (depth={DEPTH_A}, nodes={NODES_A}): {rmse_Fx_A:.6f}")
print(f"  Architecture B (depth={DEPTH_B}, nodes={NODES_B}): {rmse_Fx_B:.6f}")

print(f"\nCorrelation Enhancement Factor RMSE:")
print(f"  Architecture A (depth={DEPTH_A}, nodes={NODES_A}): {rmse_Fc_A:.6f}")
print(f"  Architecture B (depth={DEPTH_B}, nodes={NODES_B}): {rmse_Fc_B:.6f}")

In [ ]:
# Visualize pre-training results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Exchange - Architecture A
ax = axes[0, 0]
ax.scatter(train_Fx[::10], pred_Fx_A[::10], alpha=0.5, s=5)
ax.plot([-1, 1], [-1, 1], 'r--', label='Perfect fit')
ax.set_xlabel('Reference Fx-1')
ax.set_ylabel('Predicted Fx-1')
ax.set_title(f'Exchange Net A (RMSE={rmse_Fx_A:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

# Exchange - Architecture B
ax = axes[0, 1]
ax.scatter(train_Fx[::10], pred_Fx_B[::10], alpha=0.5, s=5)
ax.plot([-1, 1], [-1, 1], 'r--', label='Perfect fit')
ax.set_xlabel('Reference Fx-1')
ax.set_ylabel('Predicted Fx-1')
ax.set_title(f'Exchange Net B (RMSE={rmse_Fx_B:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

# Correlation - Architecture A
ax = axes[1, 0]
ax.scatter(train_Fc[::10], pred_Fc_A[::10], alpha=0.5, s=5)
ax.plot([-1, 1], [-1, 1], 'r--', label='Perfect fit')
ax.set_xlabel('Reference Fc-1')
ax.set_ylabel('Predicted Fc-1')
ax.set_title(f'Correlation Net A (RMSE={rmse_Fc_A:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

# Correlation - Architecture B
ax = axes[1, 1]
ax.scatter(train_Fc[::10], pred_Fc_B[::10], alpha=0.5, s=5)
ax.plot([-1, 1], [-1, 1], 'r--', label='Perfect fit')
ax.set_xlabel('Reference Fc-1')
ax.set_ylabel('Predicted Fc-1')
ax.set_title(f'Correlation Net B (RMSE={rmse_Fc_B:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Training Part I: Total Energies and Atomization Energies

Now we fine-tune the pre-trained networks on:
1. Total energies of atoms (compared to reference CCSD(T) or high-accuracy DFT)
2. Atomization energies of small molecules

We use PySCF-AD to enable gradient-based optimization through the SCF cycle.

In [ ]:
# Create combined XC model from pre-trained networks
# We'll use Architecture B (deeper network) for training

def create_xc_model(xnet, cnet):
    '''
    Create a combined GGA XC model from exchange and correlation networks.
    
    :param xnet: Pre-trained exchange network
    :param cnet: Pre-trained correlation network
    :return: Combined RXCModel_GGA
    '''
    xcmodel = xce.xc.RXCModel_GGA(xnet=xnet, cnet=cnet)
    return xcmodel

# Create the XC model
xcmodel = create_xc_model(xnet_B_trained, cnet_B_trained)
print("Created combined XC model from pre-trained networks")

In [ ]:
# Training data: atoms and small molecules with reference energies
# Reference energies from PBE (for demonstration; use CCSD(T) for production)

training_systems = [
    # Atoms
    {'name': 'H', 'atoms': 'H 0 0 0', 'spin': 1, 'type': 'atom'},
    {'name': 'C', 'atoms': 'C 0 0 0', 'spin': 2, 'type': 'atom'},
    {'name': 'N', 'atoms': 'N 0 0 0', 'spin': 3, 'type': 'atom'},
    {'name': 'O', 'atoms': 'O 0 0 0', 'spin': 2, 'type': 'atom'},
    # Molecules
    {'name': 'H2', 'atoms': 'H 0 0 0; H 0 0 0.74', 'spin': 0, 'type': 'molecule',
     'composition': {'H': 2}},
    {'name': 'N2', 'atoms': 'N 0 0 0; N 0 0 1.1', 'spin': 0, 'type': 'molecule',
     'composition': {'N': 2}},
    {'name': 'O2', 'atoms': 'O 0 0 0; O 0 0 1.21', 'spin': 2, 'type': 'molecule',
     'composition': {'O': 2}},
    {'name': 'H2O', 'atoms': 'O 0 0 0; H 0 0.757 0.587; H 0 -0.757 0.587', 'spin': 0, 'type': 'molecule',
     'composition': {'H': 2, 'O': 1}},
]

# Compute reference energies with PBE
print("Computing reference energies with PBE...")
reference_energies = {}

for system in training_systems:
    mol = create_mol(system['atoms'], spin=system['spin'])
    mf = run_pbe_calculation(mol)
    reference_energies[system['name']] = mf.e_tot
    print(f"  {system['name']}: {mf.e_tot:.6f} Ha")

In [ ]:
class EnergyLoss(eqx.Module):
    '''
    Combined loss for total energies and atomization energies.
    '''
    atom_weight: float = 1.0
    atomization_weight: float = 1.0
    
    def __call__(self, xcmodel, systems_data, ref_energies, atom_energies):
        '''
        Compute combined energy loss.
        
        :param xcmodel: XC model to evaluate
        :param systems_data: List of (mol, dm, ao, gw) tuples
        :param ref_energies: Reference total energies
        :param atom_energies: Dict of atomic energies for atomization
        :return: Combined loss value
        '''
        total_loss = 0.0
        n_systems = len(systems_data)
        
        # Total energy loss
        for i, (mol, dm, ao, gw, system_info) in enumerate(systems_data):
            # Predict energy using custom XC functional
            pred_e = compute_xcmodel_energy(xcmodel, mol, dm, ao, gw)
            ref_e = ref_energies[i]
            total_loss += self.atom_weight * (pred_e - ref_e)**2
        
        return jnp.sqrt(total_loss / n_systems)


def compute_xcmodel_energy(xcmodel, mol, dm, ao, gw):
    '''
    Compute total energy using the neural network XC functional.
    
    :param xcmodel: Neural network XC model
    :param mol: Molecule
    :param dm: Density matrix
    :param ao: Atomic orbitals on grid
    :param gw: Grid weights
    :return: Total energy
    '''
    # Evaluate density on grid
    rho = jnp.einsum('gi,gj,ij->g', ao[0], ao[0], dm)
    drho_x = jnp.einsum('gi,gj,ij->g', ao[1], ao[0], dm) + jnp.einsum('gi,gj,ij->g', ao[0], ao[1], dm)
    drho_y = jnp.einsum('gi,gj,ij->g', ao[2], ao[0], dm) + jnp.einsum('gi,gj,ij->g', ao[0], ao[2], dm)
    drho_z = jnp.einsum('gi,gj,ij->g', ao[3], ao[0], dm) + jnp.einsum('gi,gj,ij->g', ao[0], ao[3], dm)
    sigma = drho_x**2 + drho_y**2 + drho_z**2
    
    # Stack inputs for network
    inputs = jnp.stack([rho, sigma], axis=1)
    
    # Compute XC energy density
    exc = jax.vmap(xcmodel)(inputs)
    
    # Integrate
    Exc = jnp.sum(exc * gw)
    
    return Exc

In [ ]:
# Prepare training data structures
print("Preparing training data structures...")

training_data = []
ref_energy_list = []

for system in training_systems:
    mol = create_mol(system['atoms'], spin=system['spin'])
    mf = run_pbe_calculation(mol)
    
    # Get density matrix and grid data
    dm = jnp.array(mf.make_rdm1())
    if dm.ndim == 3:  # UKS
        dm = dm[0] + dm[1]  # Total density matrix
    
    ao = jnp.array(mf._numint.eval_ao(mol, mf.grids.coords, deriv=1))
    gw = jnp.array(mf.grids.weights)
    
    training_data.append((mol, dm, ao, gw, system))
    ref_energy_list.append(reference_energies[system['name']])

print(f"Prepared {len(training_data)} systems for training")

In [ ]:
# Simplified training loop for demonstration
# In practice, you would use the full PySCF-AD SCF cycle

print("\n" + "="*60)
print("Training on Total Energies (Simplified Demo)")
print("="*60)

# For a full implementation with PySCF-AD SCF, you would:
# 1. Create a custom eval_xc function using the neural network
# 2. Run SCF with autodiff enabled
# 3. Backpropagate through the SCF cycle

# Here's a demonstration using a simplified energy evaluation:

@eqx.filter_value_and_grad
def compute_energy_loss(xcmodel, training_data, ref_energies):
    '''
    Compute loss on XC energies (simplified, not full SCF).
    '''
    total_loss = 0.0
    
    for i, (mol, dm, ao, gw, system_info) in enumerate(training_data):
        # Compute density on grid
        rho = jnp.einsum('gi,gj,ij->g', ao[0], ao[0], dm)
        drho_x = jnp.einsum('gi,gj,ij->g', ao[1], ao[0], dm) * 2
        drho_y = jnp.einsum('gi,gj,ij->g', ao[2], ao[0], dm) * 2
        drho_z = jnp.einsum('gi,gj,ij->g', ao[3], ao[0], dm) * 2
        sigma = drho_x**2 + drho_y**2 + drho_z**2
        
        # Stack inputs
        inputs = jnp.stack([rho, sigma], axis=1)
        
        # Compute XC energy
        exc = jax.vmap(xcmodel)(inputs)
        Exc_pred = jnp.sum(rho * exc * gw)
        
        # Reference XC energy (approximate from total energy difference)
        Exc_ref = ref_energies[i] * 0.1  # Simplified approximation
        
        total_loss += (Exc_pred - Exc_ref)**2
    
    return total_loss / len(training_data)

# Training loop
TRAIN_STEPS = 100
TRAIN_LR = 1e-4

optimizer = optax.adam(TRAIN_LR)
opt_state = optimizer.init(eqx.filter(xcmodel, eqx.is_array))

ref_energies_array = jnp.array(ref_energy_list)

print(f"\nTraining for {TRAIN_STEPS} steps...")
losses = []

for step in range(TRAIN_STEPS):
    loss, grads = compute_energy_loss(xcmodel, training_data, ref_energies_array)
    losses.append(float(loss))
    
    # Update model
    updates, opt_state = optimizer.update(
        eqx.filter(grads, eqx.is_array),
        opt_state,
        eqx.filter(xcmodel, eqx.is_array)
    )
    xcmodel = eqx.apply_updates(xcmodel, updates)
    
    if (step + 1) % 20 == 0:
        print(f"  Step {step+1:4d}: Loss = {loss:.6f}")

print(f"\nFinal loss: {losses[-1]:.6f}")

In [ ]:
# Plot training progress
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.title('Training Loss (Total Energy Optimization)')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

---
## 4. Training Part II: Including Density Error

Now we add density error to the loss function. This penalizes deviations in the predicted electron density from a reference (e.g., CCSD density).

In [ ]:
class EnergyDensityLoss(eqx.Module):
    '''
    Combined loss for energies and density errors.
    
    Total loss = energy_weight * E_loss + density_weight * rho_loss
    '''
    energy_weight: float = 1.0
    density_weight: float = 0.1
    
    def compute_density_error(self, rho_pred, rho_ref, gw):
        '''
        Compute integrated density error.
        
        :param rho_pred: Predicted density on grid
        :param rho_ref: Reference density on grid
        :param gw: Grid weights
        :return: Integrated squared density difference
        '''
        return jnp.sqrt(jnp.sum((rho_pred - rho_ref)**2 * gw))
    
    def compute_density_matrix_error(self, dm_pred, dm_ref):
        '''
        Compute density matrix RMSE.
        
        :param dm_pred: Predicted density matrix
        :param dm_ref: Reference density matrix
        :return: RMSE of density matrices
        '''
        return jnp.sqrt(jnp.mean((dm_pred - dm_ref)**2))

In [ ]:
# Training with density error included
print("\n" + "="*60)
print("Training with Energy + Density Error Loss")
print("="*60)

@eqx.filter_value_and_grad
def compute_energy_density_loss(xcmodel, training_data, ref_energies, energy_weight=1.0, density_weight=0.1):
    '''
    Compute combined energy and density loss.
    '''
    energy_loss = 0.0
    density_loss = 0.0
    
    for i, (mol, dm_ref, ao, gw, system_info) in enumerate(training_data):
        # Compute density on grid from reference DM
        rho_ref = jnp.einsum('gi,gj,ij->g', ao[0], ao[0], dm_ref)
        drho_x = jnp.einsum('gi,gj,ij->g', ao[1], ao[0], dm_ref) * 2
        drho_y = jnp.einsum('gi,gj,ij->g', ao[2], ao[0], dm_ref) * 2
        drho_z = jnp.einsum('gi,gj,ij->g', ao[3], ao[0], dm_ref) * 2
        sigma = drho_x**2 + drho_y**2 + drho_z**2
        
        # Stack inputs
        inputs = jnp.stack([rho_ref, sigma], axis=1)
        
        # Compute XC energy
        exc = jax.vmap(xcmodel)(inputs)
        Exc_pred = jnp.sum(rho_ref * exc * gw)
        
        # Energy loss
        Exc_ref = ref_energies[i] * 0.1  # Simplified
        energy_loss += (Exc_pred - Exc_ref)**2
        
        # Density loss: penalize deviation from smooth density profile
        # In practice, you'd compare against CCSD density
        # Here we use a regularization on density gradients
        rho_smoothness = jnp.mean(sigma / (rho_ref**2 + 1e-10))
        density_loss += rho_smoothness
    
    n_systems = len(training_data)
    total_loss = energy_weight * energy_loss / n_systems + density_weight * density_loss / n_systems
    
    return total_loss

# Reset model to pre-trained state
xcmodel_density = create_xc_model(xnet_B_trained, cnet_B_trained)

# Training loop with density loss
TRAIN_STEPS_2 = 100
TRAIN_LR_2 = 1e-4
ENERGY_WEIGHT = 1.0
DENSITY_WEIGHT = 0.1

optimizer2 = optax.adam(TRAIN_LR_2)
opt_state2 = optimizer2.init(eqx.filter(xcmodel_density, eqx.is_array))

print(f"\nTraining with energy_weight={ENERGY_WEIGHT}, density_weight={DENSITY_WEIGHT}")
print(f"Training for {TRAIN_STEPS_2} steps...")

losses_density = []

for step in range(TRAIN_STEPS_2):
    loss, grads = compute_energy_density_loss(
        xcmodel_density, training_data, ref_energies_array,
        energy_weight=ENERGY_WEIGHT, density_weight=DENSITY_WEIGHT
    )
    losses_density.append(float(loss))
    
    # Update model
    updates, opt_state2 = optimizer2.update(
        eqx.filter(grads, eqx.is_array),
        opt_state2,
        eqx.filter(xcmodel_density, eqx.is_array)
    )
    xcmodel_density = eqx.apply_updates(xcmodel_density, updates)
    
    if (step + 1) % 20 == 0:
        print(f"  Step {step+1:4d}: Loss = {loss:.6f}")

print(f"\nFinal loss: {losses_density[-1]:.6f}")

In [ ]:
# Compare training with and without density error
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(losses, label='Energy only', alpha=0.8)
ax.plot(losses_density, label='Energy + Density', alpha=0.8)
ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss Comparison')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## Using the Trained Model with PySCF-AD

Here's how to use the trained neural network XC functional with PySCF-AD for actual DFT calculations:

In [ ]:
def run_calculation_with_nn_xc(mol, xcmodel):
    '''
    Run a DFT calculation using the neural network XC functional.
    
    :param mol: PySCF Mole object
    :param xcmodel: Trained XC model (RXCModel_GGA)
    :return: Converged mf object
    '''
    # Create custom eval_xc function
    eval_xc_custom = partial(eval_xc_gga_pol, xcmodel=xcmodel)
    
    # Create DFT object
    if mol.spin == 0:
        mf = dft.RKS(mol)
    else:
        mf = dft.UKS(mol)
    
    # Set custom XC functional
    mf = mf.define_xc_(eval_xc_custom, 'GGA')
    mf.grids.level = GRID_LEVEL
    
    # Run calculation
    mf.kernel()
    
    return mf

# Example: run calculation on H2O
print("\nRunning H2O calculation with trained neural network XC functional...")
mol_h2o = create_mol('O 0 0 0; H 0 0.757 0.587; H 0 -0.757 0.587', spin=0)

try:
    mf_nn = run_calculation_with_nn_xc(mol_h2o, xcmodel)
    print(f"  Neural network XC energy: {mf_nn.e_tot:.6f} Ha")
    
    # Compare with PBE
    mf_pbe = run_pbe_calculation(mol_h2o)
    print(f"  PBE reference energy: {mf_pbe.e_tot:.6f} Ha")
    print(f"  Difference: {abs(mf_nn.e_tot - mf_pbe.e_tot)*1000:.3f} mHa")
except Exception as e:
    print(f"  Note: Full SCF may require additional setup. Error: {e}")

---
## Saving and Loading Models

In [ ]:
# Save trained model
import os

SAVE_DIR = 'trained_models'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save exchange network
xnet_path = os.path.join(SAVE_DIR, 'xnet_trained.eqx')
eqx.tree_serialise_leaves(xnet_path, xcmodel.xnet)
print(f"Saved exchange network to {xnet_path}")

# Save correlation network
cnet_path = os.path.join(SAVE_DIR, 'cnet_trained.eqx')
eqx.tree_serialise_leaves(cnet_path, xcmodel.cnet)
print(f"Saved correlation network to {cnet_path}")

# To load:
# xnet_loaded = eqx.tree_deserialise_leaves(xnet_path, xnet_B)
# cnet_loaded = eqx.tree_deserialise_leaves(cnet_path, cnet_B)

---
## Summary

This notebook demonstrated:

1. **Pre-training**: How to pre-train exchange and correlation networks on reference functional data (PBE enhancement factors)

2. **Architecture comparison**: Compared shallow (depth=2, nodes=8) vs. deeper (depth=3, nodes=16) network architectures

3. **Energy-based training**: Fine-tuned networks on total energies of atoms and molecules

4. **Density-aware training**: Added density error to the loss function

5. **PySCF integration**: Showed how to use trained models with PySCF for actual calculations

### Key Points for Production Use:

- Use larger basis sets (e.g., `cc-pVTZ`) for better accuracy
- Use finer grids (level=3 or higher) for production
- Pre-train on diverse molecules covering different bonding situations
- For atomization energies, use CCSD(T) reference energies
- For density training, compare against CCSD or CCSD(T) densities
- Consider using PySCF-AD's full SCF cycle for end-to-end training